In [1]:
from dataclasses import dataclass
from typing import Set, List
import json


@dataclass
class Rule:
    rule_id: str
    conditions: Set[str]
    conclusion: str
    advice: str
    explanation: str
    priority: int = 1
    confidence: float = 1.0

    def matches(self, facts):
        return self.conditions.issubset(facts)


class KnowledgeBase:

    def __init__(self):
        self.rules = []

    def add_rule(self, rule):
        self.rules.append(rule)

    def get_rules_for_goal(self, goal):
        return [
            rule for rule in self.rules
            if rule.conclusion == goal
        ]

    def show_rules(self):
        print("\nKnowledge Base")
        print("-" * 70)

        for rule in self.rules:
            print(f"\n{rule.rule_id}")
            print("IF   :", " AND ".join(sorted(rule.conditions)))
            print("THEN :", rule.conclusion)
            print("Advice:", rule.advice)
            print("Priority:", rule.priority)
            print("Confidence:", f"{rule.confidence:.0%}")


kb = KnowledgeBase()


kb.add_rule(Rule(
    "R1",
    {"laptop_not_turning_on"},
    "power_problem",
    "Check the charger, power cable and battery.",
    "The laptop is not turning on, so there may be a power problem.",
    10,
    0.95
))

kb.add_rule(Rule(
    "R2",
    {"laptop_not_turning_on", "charger_connected"},
    "possible_battery_problem",
    "Try another charger or check the battery.",
    "The charger is connected but the laptop still does not turn on.",
    11,
    0.92
))

kb.add_rule(Rule(
    "R3",
    {"laptop_on", "internet_not_working"},
    "network_problem",
    "Check Wi-Fi settings and restart the router.",
    "The laptop is on but internet connectivity is unavailable.",
    10,
    0.95
))

kb.add_rule(Rule(
    "R4",
    {"network_problem", "wifi_enabled"},
    "router_or_isp_problem",
    "Restart the router and check the Internet Service Provider.",
    "Wi-Fi is enabled but network connectivity is still unavailable.",
    9,
    0.90
))

kb.add_rule(Rule(
    "R5",
    {"laptop_on", "laptop_slow"},
    "performance_problem",
    "Close unused programs and restart the laptop.",
    "The laptop is working but its performance is slow.",
    8,
    0.90
))

kb.add_rule(Rule(
    "R6",
    {"laptop_slow", "high_cpu_usage"},
    "cpu_overload",
    "Close CPU-intensive applications and check Task Manager.",
    "High CPU usage together with slow performance indicates CPU overload.",
    10,
    0.94
))

kb.add_rule(Rule(
    "R7",
    {"laptop_slow", "low_disk_space"},
    "storage_problem",
    "Delete unnecessary files and free up disk space.",
    "Low disk space can affect system performance.",
    9,
    0.91
))

kb.add_rule(Rule(
    "R8",
    {"laptop_slow", "laptop_hot"},
    "thermal_problem",
    "Clean the cooling vents and make sure the laptop has proper airflow.",
    "Slow performance together with high temperature suggests a thermal problem.",
    10,
    0.93
))

kb.add_rule(Rule(
    "R9",
    {"thermal_problem", "fan_noisy"},
    "cooling_system_problem",
    "Inspect the cooling fan and clean the cooling system.",
    "A thermal problem with unusual fan noise suggests a cooling-system issue.",
    11,
    0.94
))

kb.add_rule(Rule(
    "R10",
    {"laptop_on", "screen_black"},
    "display_problem",
    "Check the display connection or try an external monitor.",
    "The laptop is powered on but the screen is black.",
    10,
    0.91
))

kb.add_rule(Rule(
    "R11",
    {"laptop_on", "keyboard_not_working"},
    "keyboard_problem",
    "Check the keyboard connection and keyboard drivers.",
    "The laptop is on but keyboard input is unavailable.",
    8,
    0.88
))

kb.add_rule(Rule(
    "R12",
    {"laptop_slow", "unknown_background_process"},
    "possible_malware_problem",
    "Run a trusted security scan and inspect unfamiliar processes.",
    "Unknown background activity with poor performance may indicate a security issue.",
    10,
    0.82
))


class InferenceEngine:

    def __init__(self, knowledge_base):
        self.kb = knowledge_base
        self.trace = []

    def forward_chaining(self, facts):

        facts = set(facts)
        self.trace = []

        changed = True
        iteration = 0

        while changed:

            changed = False
            iteration += 1

            matching_rules = [
                rule for rule in self.kb.rules
                if rule.matches(facts)
                and rule.conclusion not in facts
            ]

            matching_rules.sort(
                key=lambda r: (
                    r.priority,
                    r.confidence,
                    len(r.conditions)
                ),
                reverse=True
            )

            for rule in matching_rules:

                facts.add(rule.conclusion)
                changed = True

                self.trace.append({
                    "iteration": iteration,
                    "rule": rule.rule_id,
                    "conditions": list(rule.conditions),
                    "conclusion": rule.conclusion,
                    "confidence": rule.confidence
                })

        return facts

    def backward_chaining(self, goal, facts, visited=None, level=0):

        if visited is None:
            visited = set()

        if goal in facts:
            print("  " * level + "Fact found:", goal)
            return True

        if goal in visited:
            return False

        visited.add(goal)

        print("  " * level + "Trying to prove:", goal)

        rules = self.kb.get_rules_for_goal(goal)

        rules.sort(
            key=lambda r: (
                r.priority,
                r.confidence
            ),
            reverse=True
        )

        for rule in rules:

            print(
                "  " * level +
                f"Checking {rule.rule_id}"
            )

            possible = True

            for condition in rule.conditions:

                if not self.backward_chaining(
                    condition,
                    facts,
                    visited,
                    level + 1
                ):
                    possible = False
                    break

            if possible:
                print(
                    "  " * level +
                    "Goal proved:",
                    goal
                )
                return True

        print(
            "  " * level +
            "Goal not proved:",
            goal
        )

        return False


class ExplanationEngine:

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

    def explain(self, conclusion):

        rules = self.kb.get_rules_for_goal(conclusion)

        for rule in rules:

            print(f"\nRule: {rule.rule_id}")
            print(
                "Conditions:",
                " AND ".join(sorted(rule.conditions))
            )
            print("Conclusion:", rule.conclusion)
            print("Reason:", rule.explanation)
            print(
                "Confidence:",
                f"{rule.confidence:.0%}"
            )


class RecommendationEngine:

    def __init__(self, knowledge_base):
        self.kb = knowledge_base

    def get_recommendations(self, facts):
        results = []

        for rule in self.kb.rules:

            if rule.conclusion in facts:

                score = (
                    rule.priority *
                    rule.confidence *
                    len(rule.conditions)
                )

                results.append({
                    "rule": rule.rule_id,
                    "problem": rule.conclusion,
                    "advice": rule.advice,
                    "confidence": rule.confidence,
                    "priority": rule.priority,
                    "score": score
                })

        results.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        return results


inference = InferenceEngine(kb)
explainer = ExplanationEngine(kb)
recommendation = RecommendationEngine(kb)


def run_diagnosis():

    print("\nLaptop Troubleshooting Expert System")
    print("-" * 50)

    print("\nAnswer using yes or no.\n")

    questions = {
        "laptop_on":
            "Is the laptop turned on?",
        "internet_not_working":
            "Is the internet not working?",
        "wifi_enabled":
            "Is Wi-Fi enabled?",
        "laptop_slow":
            "Is the laptop slow?",
        "high_cpu_usage":
            "Is CPU usage high?",
        "low_disk_space":
            "Is disk space low?",
        "laptop_hot":
            "Is the laptop unusually hot?",
        "fan_noisy":
            "Is the fan unusually noisy?",
        "screen_black":
            "Is the screen black?",
        "keyboard_not_working":
            "Is the keyboard not working?",
        "unknown_background_process":
            "Is there an unknown background process?",
        "charger_connected":
            "Is the charger connected?",
        "laptop_not_turning_on":
            "Is the laptop not turning on?"
    }

    facts = set()

    for name, question in questions.items():

        answer = input(question + " ").strip().lower()

        while answer not in ["yes", "no"]:
            print("Please enter yes or no.")
            answer = input(question + " ").strip().lower()

        if answer == "yes":
            facts.add(name)

    print("\nInitial Facts")
    print("-" * 50)

    for fact in sorted(facts):
        print(fact)

    final_facts = inference.forward_chaining(facts)

    derived_facts = final_facts - facts

    print("\nDerived Facts")
    print("-" * 50)

    if derived_facts:
        for fact in sorted(derived_facts):
            print(fact)
    else:
        print("No new facts were derived.")

    print("\nInference Trace")
    print("-" * 50)

    for step in inference.trace:

        print(
            f"Iteration {step['iteration']} | "
            f"{step['rule']} | "
            f"{step['conclusion']} | "
            f"Confidence: {step['confidence']:.0%}"
        )

    results = recommendation.get_recommendations(
        final_facts
    )

    print("\nRecommendations")
    print("-" * 50)

    if results:

        for i, result in enumerate(results[:5], 1):

            print(f"\n{i}. {result['problem']}")
            print("   Rule:", result["rule"])
            print("   Advice:", result["advice"])
            print(
                "   Confidence:",
                f"{result['confidence']:.0%}"
            )
            print(
                "   Priority:",
                result["priority"]
            )

    else:

        print(
            "No matching troubleshooting rule was found."
        )

    return facts, final_facts


# Display the rules
kb.show_rules()


# Run the expert system
initial_facts, final_facts = run_diagnosis()


# Explanation of derived conclusions
print("\nExplanations")
print("-" * 50)

for fact in sorted(final_facts - initial_facts):

    explainer.explain(fact)


# Backward chaining example
print("\nBackward Chaining")
print("-" * 50)

goal = "cooling_system_problem"

print("Goal:", goal)

proved = inference.backward_chaining(
    goal,
    initial_facts
)

print(
    "\nResult:",
    "PROVED" if proved else "NOT PROVED"
)


# Save knowledge base
knowledge_base_data = []

for rule in kb.rules:

    knowledge_base_data.append({
        "rule_id": rule.rule_id,
        "conditions": list(rule.conditions),
        "conclusion": rule.conclusion,
        "advice": rule.advice,
        "explanation": rule.explanation,
        "priority": rule.priority,
        "confidence": rule.confidence
    })


with open(
    "laptop_knowledge_base.json",
    "w"
) as file:

    json.dump(
        knowledge_base_data,
        file,
        indent=4
    )


print(
    "\nKnowledge base saved as "
    "laptop_knowledge_base.json"
)


Knowledge Base
----------------------------------------------------------------------

R1
IF   : laptop_not_turning_on
THEN : power_problem
Advice: Check the charger, power cable and battery.
Priority: 10
Confidence: 95%

R2
IF   : charger_connected AND laptop_not_turning_on
THEN : possible_battery_problem
Advice: Try another charger or check the battery.
Priority: 11
Confidence: 92%

R3
IF   : internet_not_working AND laptop_on
THEN : network_problem
Advice: Check Wi-Fi settings and restart the router.
Priority: 10
Confidence: 95%

R4
IF   : network_problem AND wifi_enabled
THEN : router_or_isp_problem
Advice: Restart the router and check the Internet Service Provider.
Priority: 9
Confidence: 90%

R5
IF   : laptop_on AND laptop_slow
THEN : performance_problem
Advice: Close unused programs and restart the laptop.
Priority: 8
Confidence: 90%

R6
IF   : high_cpu_usage AND laptop_slow
THEN : cpu_overload
Advice: Close CPU-intensive applications and check Task Manager.
Priority: 10
Confid